In [1]:
!pip install scrapy scrapy-playwright playwright playwright-stealth nest-asyncio -q
!playwright install chromium
!playwright install-deps chromium
import nest_asyncio
nest_asyncio.apply()

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done


In [2]:
!playwright --version

Version 1.58.0


In [3]:
%%writefile settings.py

DOWNLOAD_HANDLERS = {
    "http": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
    "https": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
}

# Reactor bắt buộc để chạy Playwright (Asyncio)
TWISTED_REACTOR = "twisted.internet.asyncioreactor.AsyncioSelectorReactor"

PLAYWRIGHT_LAUNCH_OPTIONS = {
    "headless": True,  # Đổi thành False nếu muốn nhìn thấy trình duyệt chạy
    "args": [
        "--no-sandbox",
        "--disable-setuid-sandbox",
        "--disable-dev-shm-usage",
        "--disable-blink-features=AutomationControlled", # Ẩn dấu hiệu Robot
    ],
}

# Giả lập thông số màn hình và User-Agent
PLAYWRIGHT_CONTEXT_ARGS = {
    "user_agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "viewport": {"width": 1920, "height": 1080},
}

ROBOTSTXT_OBEY = False
CONCURRENT_REQUESTS = 2
DOWNLOAD_DELAY = 3
TELNETCONSOLE_ENABLED = False # Tắt để tránh lỗi port trên Notebook

Overwriting settings.py


In [4]:
%%writefile myspider.py
import scrapy
from scrapy_playwright.page import PageMethod
from datetime import datetime

class ItViecSpider(scrapy.Spider):
    name = "itviec"

    custom_settings = {
        "TWISTED_REACTOR": "twisted.internet.asyncioreactor.AsyncioSelectorReactor",
        "DOWNLOAD_HANDLERS": {
            "https": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
            "http": "scrapy_playwright.handler.ScrapyPlaywrightDownloadHandler",
        },
        "PLAYWRIGHT_LAUNCH_OPTIONS": {"headless": True},
        "USER_AGENT": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "ROBOTSTXT_OBEY": False,
        "CONCURRENT_REQUESTS": 2,
        "DOWNLOAD_DELAY": 2,
    }

    def start_requests(self):
        base_url = "https://itviec.com/it-jobs"
        for page in range(1, 16):
            url = f"{base_url}?page={page}"
            yield scrapy.Request(
                url=url,
                meta={
                    "playwright": True,
                    "playwright_page_methods": [
                        PageMethod("wait_for_selector", "h3[data-search--job-selection-target='jobTitle']"),
                    ],
                },
                callback=self.parse
            )

    def parse(self, response):
        job_title_elements = response.css("h3[data-search--job-selection-target='jobTitle']")
        self.logger.info(f"Tìm thấy {len(job_title_elements)} jobs tại {response.url}")

        for element in job_title_elements:
            job_url = element.css("::attr(data-url)").get()
            title   = element.css("::text").get(default="").strip()
            company = element.xpath(
                "ancestor::div[contains(@class,'d-flex')]//span[contains(@class,'ims-2')]//a//text()"
            ).get(default="").strip()

            if job_url:
                yield scrapy.Request(
                    url=response.urljoin(job_url),
                    meta={
                        "playwright": True,
                        "playwright_page_methods": [
                            PageMethod("wait_for_selector", ".job-show-info", timeout=15000),
                        ],
                        "title":        title,
                    },
                    callback=self.parse_job_details
                )

    def parse_job_details(self, response):
        title        = response.meta.get("title", "N/A")

        # Lấy tất cả text trong job-show-info, BỎ QUA thẻ có class "jd-photos"
        job_preview_texts = response.xpath(
            "//*[contains(@class,'job-show-info')]"
            "//*[not(ancestor-or-self::*[contains(@class,'jd-photos')])]/text()"
        ).getall()
        job_preview = "\n".join(line.strip() for line in job_preview_texts if line.strip())

        # Lấy mô tả chi tiết công việc
        job_detail_texts = response.css("section.job-content ::text").getall()
        job_detail = "\n".join(line.strip() for line in job_detail_texts if line.strip())

        # Lấy thông tin công ty
        company_info_texts = response.css("section.job-show-employer-info ::text").getall()
        company_info = "\n".join(line.strip() for line in company_info_texts if line.strip())

        yield {
            "title":        title,
            "url":          response.url,
            "job_preview":  job_preview,
            "job_detail":   job_detail,
            "company_info": company_info,
            "crawled_at":   datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }

Overwriting myspider.py


In [5]:
# Xóa file cũ nếu có
!rm -f results.json

# Chạy spider, lưu output ra file JSON
!scrapy runspider myspider.py -o results.json --loglevel=INFO

# Xem kết quả (giới hạn để không bị tràn output)
!cat results.json

2026-04-07 15:58:59 [scrapy.utils.log] INFO: Scrapy 2.14.2 started (bot: scrapybot)
2026-04-07 15:58:59 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '25.5.0',
 'Python': '3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]',
 'pyOpenSSL': '24.2.1 (OpenSSL 3.3.2 3 Sep 2024)',
 'cryptography': '43.0.3',
 'Platform': 'Linux-6.6.113+-x86_64-with-glibc2.35'}
2026-04-07 15:58:59 [scrapy.addons] INFO: Enabled addons:
[]
2026-04-07 15:58:59 [scrapy.extensions.telnet] INFO: Telnet Password: 2ea4fc6717addfcc
2026-04-07 15:58:59 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCount',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2026-04-07 15:58:59 [scrapy.crawler] INFO: Overridden settings:
{'CONCUR

In [6]:
import pandas as pd
import os

if os.path.exists("results.json") and os.path.getsize("results.json") > 0:
    df = pd.read_json("results.json")
    display(df)
else:
    print("RẤT TIẾC: Vẫn chưa lấy được dữ liệu. ITviec đang chặn IP Kaggle cực gắt.")

,title,url,job_preview,job_detail,company_info,crawled_at
0,Site Reliability Engineer (DevOps/Azure/Linux ),https://itviec.com/it-jobs/site-reliability-en...,"165 Thái Hà, Dong Da, Ha Noi\nAt office\nPoste...",Top 3 reasons to join us\nGreat international ...,Optimizely\n4.6\nWe’re on a mission to help pe...,2026-04-07 15:59:09
1,Junior/Senior Fullstack Developer (React/Golang ),https://itviec.com/it-jobs/junior-senior-fulls...,"19F, Coninco Tower 4 Tôn Thất Tùng, Dong Da, H...","Top 3 reasons to join us\n13th bonus salary, p...",8Seneca\n4.5\nPure play IT team extensions B2b...,2026-04-07 15:59:10
2,Senior Machine Learning Engineer – Physics-inf...,https://itviec.com/it-jobs/senior-machine-lear...,"8-10 Tạ Hiện, P. Thạnh Mỹ Lợi, TP. Thủ Đức, TP...",Top 3 reasons to join us\nWork on challenging ...,Koidra Tech\nKoidra Tech\nCompany type\nIT Pro...,2026-04-07 15:59:13
3,Middle/ Senior Mobile Developer (Flutter),https://itviec.com/it-jobs/middle-senior-mobil...,"Tower 2 (T26) Times City, 458 Minh Khai, Hai B...",Top 3 reasons to join us\nMôi trường đẩy nhanh...,One Mount Group\n4.2\nOne Mount\nCompany type\...,2026-04-07 15:59:15
4,Technical Business Analyst,https://itviec.com/it-jobs/technical-business-...,"Waseco Building - 10 Pho Quang Street, Ward 02...",Top 3 reasons to join us\nFlexible Friday afte...,Pizza Hut Digital & Technology\n4.8\nPizza Hut...,2026-04-07 15:59:18
...,...,...,...,...,...,...
295,Embedded Engineer (AUTOSAR Integrator/C++/Tester),https://itviec.com/it-jobs/embedded-engineer-a...,"364 Cong Hoa street, ward 13, Tan Binh, Ho Chi...",Top 3 reasons to join us\nCommitted 13th-mth b...,Bosch Global Software Technologies Company Lim...,2026-04-07 16:11:41
296,Automotive Embedded Developer (Flash Bootloade...,https://itviec.com/it-jobs/automotive-embedded...,"364 Cong Hoa street, ward 13, Tan Binh, Ho Chi...",Top 3 reasons to join us\nCommitted 13th-mth b...,Bosch Global Software Technologies Company Lim...,2026-04-07 16:11:44
297,Senior Embedded Engineer (C/C++),https://itviec.com/it-jobs/senior-embedded-eng...,"364 Cong Hoa street, ward 13, Tan Binh, Ho Chi...",Top 3 reasons to join us\nCommitted 13th-mth b...,Bosch Global Software Technologies Company Lim...,2026-04-07 16:11:47
298,"FullStack Developer (.Net, C#, NextJS, ReactJS)",https://itviec.com/it-jobs/fullstack-developer...,Công ty Cổ phần Ứng dụng công nghệ Logistics (...,Top 3 reasons to join us\nMức lương cạnh tranh...,Logistics Technology Application (LTA)\nLogist...,2026-04-07 16:11:49
